# HPO with Optuna Integration

## Overview

This notebook demonstrates how SpaX integrates seamlessly with Optuna for hyperparameter optimization. **All SpaX features** - conditional spaces, nested configs, and polymorphic fields - work automatically with Optuna's optimization algorithms.

**What you'll learn:**
- Basic SpaX + Optuna integration pattern
- Conditional parameters with Optuna (automatic dependency handling)
- Nested configurations in HPO
- Polymorphic fields (different config types) with Optuna
- Complete workflow with best practices

**Prerequisites:**
- Basic understanding of SpaX `Config` (see notebook 00)
- Familiarity with conditional parameters (see notebook 01)
- Familiarity with nested configs (helpful, see notebook 02)
- Optuna installed: `pip install optuna`

**What this notebook is NOT:**
- ❌ A tutorial on Optuna's advanced features (pruning, samplers, multi-objective)
- ❌ A guide to hyperparameter tuning strategies
- ✅ A demonstration of SpaX's seamless Optuna integration

**Why this matters:**
- 🚀 Zero boilerplate - `Config.from_trial(trial)` is all you need
- 🔒 Type-safe HPO - invalid configs never get suggested
- 🧩 Complex spaces - conditionals, nesting, polymorphism all work automatically
- 📊 Standard Optuna workflow - use all Optuna's tools and visualizations

Let's start with the simplest example.

In [1]:
# Basic Optuna integration - the simplest example
import optuna

import spax as sp


# Simple configuration for optimization
class SimpleConfig(sp.Config):
    """Simple configuration with a few parameters."""

    learning_rate: float = sp.Float(ge=1e-5, le=1e-1, distribution="log")
    batch_size: int = sp.Int(ge=16, le=128)
    num_layers: int = sp.Int(ge=1, le=5)
    dropout: float = sp.Float(ge=0.0, le=0.5)


# Objective function - simulates model training
def objective(trial):
    """
    Objective function for Optuna.

    This is where SpaX integration happens:
    - Config.from_trial(trial) creates a config from Optuna's suggestions
    - All parameters are automatically suggested based on their spaces
    """
    # Create config from trial - this is the magic line!
    config = SimpleConfig.from_trial(trial)

    # Simulate "training" with a synthetic score
    # In real code, this would be: train_model(config) -> accuracy
    score = simulate_training(config)

    return score


def simulate_training(config):
    """
    Synthetic scoring function that mimics ML training.
    Returns a score in [0, 1] based on config values.
    """
    # Optimal values (hidden from optimizer)
    optimal_lr = 0.001
    optimal_batch = 32
    optimal_layers = 3
    optimal_dropout = 0.2

    # Score decreases with distance from optimal values
    lr_score = 1.0 - abs(config.learning_rate - optimal_lr) / optimal_lr
    batch_score = 1.0 - abs(config.batch_size - optimal_batch) / 100
    layer_score = 1.0 - abs(config.num_layers - optimal_layers) / 5
    dropout_score = 1.0 - abs(config.dropout - optimal_dropout) / 0.5

    # Combined score with some noise
    score = (lr_score + batch_score + layer_score + dropout_score) / 4
    return max(0.0, min(1.0, score))


# Run optimization
print("🔍 Running Optuna optimization with SpaX config...\n")

study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=20, show_progress_bar=False)

# Show results
print("✅ Optimization complete!")
print(f"   Best score: {study.best_value:.4f}")
print("\n📊 Best configuration:")
print(f"   learning_rate: {study.best_params['SimpleConfig.learning_rate']:.6f}")
print(f"   batch_size: {study.best_params['SimpleConfig.batch_size']}")
print(f"   num_layers: {study.best_params['SimpleConfig.num_layers']}")
print(f"   dropout: {study.best_params['SimpleConfig.dropout']:.3f}")

print("\n" + "=" * 60)
print(
    "\n💡 Notice: All parameters have hierarchical names like 'SimpleConfig.learning_rate'"
)
print("   This prevents naming conflicts in complex configs.")

[I 2025-10-26 21:47:46,434] A new study created in memory with name: no-name-e49a6f7a-96ea-43d2-b7f7-1ff69ed7a45f
[I 2025-10-26 21:47:46,435] Trial 0 finished with value: 0.5015581701496624 and parameters: {'SimpleConfig.learning_rate': 0.00031489116479568613, 'SimpleConfig.batch_size': 123, 'SimpleConfig.num_layers': 4, 'SimpleConfig.dropout': 0.2993292420985183}. Best is trial 0 with value: 0.5015581701496624.
[I 2025-10-26 21:47:46,435] Trial 1 finished with value: 0.5414759352302828 and parameters: {'SimpleConfig.learning_rate': 4.207988669606632e-05, 'SimpleConfig.batch_size': 33, 'SimpleConfig.num_layers': 1, 'SimpleConfig.dropout': 0.4330880728874676}. Best is trial 1 with value: 0.5414759352302828.
[I 2025-10-26 21:47:46,436] Trial 2 finished with value: 0.21306865989308543 and parameters: {'SimpleConfig.learning_rate': 0.002537815508265664, 'SimpleConfig.batch_size': 96, 'SimpleConfig.num_layers': 1, 'SimpleConfig.dropout': 0.48495492608099716}. Best is trial 1 with value: 0.5

🔍 Running Optuna optimization with SpaX config...

✅ Optimization complete!
   Best score: 0.8507

📊 Best configuration:
   learning_rate: 0.001176
   batch_size: 42
   num_layers: 4
   dropout: 0.260


💡 Notice: All parameters have hierarchical names like 'SimpleConfig.learning_rate'
   This prevents naming conflicts in complex configs.


In [2]:
# Conditional parameters work automatically with Optuna
class ConditionalConfig(sp.Config):
    """Configuration with conditional parameters."""

    use_regularization: bool = sp.Categorical([True, False])

    # L2 weight decay - only relevant when regularization is enabled
    weight_decay: float = sp.Conditional(
        sp.FieldCondition("use_regularization", sp.EqualsTo(True)),
        true=sp.Float(ge=1e-5, le=1e-1, distribution="log"),
        false=0.0,
    )

    optimizer: str = sp.Categorical(["adam", "sgd", "rmsprop"])

    # Momentum - only for SGD
    momentum: float = sp.Conditional(
        sp.FieldCondition("optimizer", sp.EqualsTo("sgd")),
        true=sp.Float(ge=0.0, le=0.99),
        false=0.0,
    )

    learning_rate: float = sp.Float(ge=1e-5, le=1e-2, distribution="log")


def conditional_objective(trial):
    """Objective with conditional parameters."""
    config = ConditionalConfig.from_trial(trial)

    # Simulate training
    score = 0.6
    score += (
        0.1 if config.use_regularization and 1e-4 < config.weight_decay < 1e-2 else 0
    )
    score += 0.15 if config.optimizer == "adam" else 0
    score += 0.1 if config.optimizer == "sgd" and config.momentum > 0.8 else 0
    score += 0.15 if 1e-4 < config.learning_rate < 1e-3 else 0

    return score


# Run optimization
print("🔍 Optimizing config with conditional parameters...\n")

study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=100)
)
study.optimize(conditional_objective, n_trials=30, show_progress_bar=False)

print(f"✅ Best score: {study.best_value:.4f}\n")
print("📊 Best parameters:")
for key, value in study.best_params.items():
    param_name = key.split(".")[-1]  # Get just the parameter name
    if isinstance(value, float):
        print(f"   {param_name}: {value:.6f}")
    else:
        print(f"   {param_name}: {value}")

print("\n" + "=" * 60)
print("\n🔒 Validation: Let's check a few trials to verify conditional constraints...")

# Check that conditional constraints are always satisfied
valid_trials = 0
for trial in study.trials[:10]:  # Check first 10 trials
    params = trial.params

    # Extract values (handling the hierarchical naming)
    use_reg = params.get("ConditionalConfig.use_regularization", None)
    weight_decay = params.get("ConditionalConfig.weight_decay::true_branch", 0.0)
    optimizer = params.get("ConditionalConfig.optimizer", None)
    momentum = params.get("ConditionalConfig.momentum::true_branch", 0.0)

    # Check constraints
    if not use_reg:
        assert weight_decay == 0.0, (
            "weight_decay should be 0 when use_regularization=False"
        )

    if optimizer != "sgd":
        assert momentum == 0.0, "momentum should be 0 when optimizer != sgd"

    valid_trials += 1

print(f"   ✅ All {valid_trials} trials satisfy conditional constraints!")
print("   Optuna never suggested invalid parameter combinations.")

[I 2025-10-26 21:48:49,046] A new study created in memory with name: no-name-fb7b50f1-8c92-4892-8a32-d2498d9868e4
[I 2025-10-26 21:48:49,049] Trial 0 finished with value: 0.75 and parameters: {'ConditionalConfig.learning_rate': 0.00042679058013626766, 'ConditionalConfig.optimizer': 'rmsprop', 'ConditionalConfig.use_regularization': 'False'}. Best is trial 0 with value: 0.75.
[I 2025-10-26 21:48:49,052] Trial 1 finished with value: 0.75 and parameters: {'ConditionalConfig.learning_rate': 0.0010286017388981877, 'ConditionalConfig.optimizer': 'adam', 'ConditionalConfig.use_regularization': 'True', 'ConditionalConfig.weight_decay::true_branch': 5.512046570407665e-05}. Best is trial 0 with value: 0.75.
[I 2025-10-26 21:48:49,055] Trial 2 finished with value: 0.7 and parameters: {'ConditionalConfig.learning_rate': 2.1141250463248296e-05, 'ConditionalConfig.optimizer': 'sgd', 'ConditionalConfig.momentum::true_branch': 0.17022160260526828, 'ConditionalConfig.use_regularization': 'True', 'Condi

🔍 Optimizing config with conditional parameters...



[I 2025-10-26 21:48:49,257] Trial 27 finished with value: 0.6 and parameters: {'ConditionalConfig.learning_rate': 4.008203016999779e-05, 'ConditionalConfig.optimizer': 'rmsprop', 'ConditionalConfig.use_regularization': 'True', 'ConditionalConfig.weight_decay::true_branch': 0.017495506799232098}. Best is trial 13 with value: 1.0.
[I 2025-10-26 21:48:49,269] Trial 28 finished with value: 1.0 and parameters: {'ConditionalConfig.learning_rate': 0.0004761833487243147, 'ConditionalConfig.optimizer': 'adam', 'ConditionalConfig.use_regularization': 'True', 'ConditionalConfig.weight_decay::true_branch': 0.0009618938913398871}. Best is trial 13 with value: 1.0.
[I 2025-10-26 21:48:49,280] Trial 29 finished with value: 0.85 and parameters: {'ConditionalConfig.learning_rate': 0.00018452962742350397, 'ConditionalConfig.optimizer': 'rmsprop', 'ConditionalConfig.use_regularization': 'True', 'ConditionalConfig.weight_decay::true_branch': 0.003912735361319053}. Best is trial 13 with value: 1.0.


✅ Best score: 1.0000

📊 Best parameters:
   learning_rate: 0.000175
   optimizer: adam
   use_regularization: True
   weight_decay::true_branch: 0.004599


🔒 Validation: Let's check a few trials to verify conditional constraints...
   ✅ All 10 trials satisfy conditional constraints!
   Optuna never suggested invalid parameter combinations.


In [3]:
# Nested configurations - hierarchical parameter spaces
class OptimizerConfig(sp.Config):
    """Optimizer sub-configuration."""

    name: str = sp.Categorical(["adam", "sgd"])
    learning_rate: float = sp.Float(ge=1e-5, le=1e-2, distribution="log")
    weight_decay: float = sp.Float(ge=0.0, le=0.1)


class ModelConfig(sp.Config):
    """Model architecture sub-configuration."""

    num_layers: int = sp.Int(ge=2, le=6)
    hidden_dim: int = sp.Int(ge=64, le=256)
    activation: str = sp.Categorical(["relu", "gelu"])


class TrainingConfig(sp.Config):
    """Complete training configuration with nested configs."""

    model: ModelConfig
    optimizer: OptimizerConfig
    batch_size: int = sp.Int(ge=16, le=128)


def nested_objective(trial):
    """Objective with nested configurations."""
    config = TrainingConfig.from_trial(trial)

    # Simulate training with nested parameters
    score = 0.5

    # Model architecture scoring
    score += 0.1 if config.model.num_layers == 4 else 0
    score += 0.1 if 128 <= config.model.hidden_dim <= 192 else 0
    score += 0.05 if config.model.activation == "gelu" else 0

    # Optimizer scoring
    score += 0.1 if config.optimizer.name == "adam" else 0
    score += 0.1 if 1e-4 < config.optimizer.learning_rate < 1e-3 else 0
    score += 0.05 if config.optimizer.weight_decay < 0.01 else 0

    return score


# Run optimization
print("🔍 Optimizing nested configuration...\n")

study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=200)
)
study.optimize(nested_objective, n_trials=40, show_progress_bar=False)

print(f"✅ Best score: {study.best_value:.4f}\n")
print("📊 Best nested configuration:")
print("\n   Model:")
print(
    f"      num_layers: {study.best_params['TrainingConfig.model::ModelConfig.num_layers']}"
)
print(
    f"      hidden_dim: {study.best_params['TrainingConfig.model::ModelConfig.hidden_dim']}"
)
print(
    f"      activation: {study.best_params['TrainingConfig.model::ModelConfig.activation']}"
)
print("\n   Optimizer:")
print(
    f"      name: {study.best_params['TrainingConfig.optimizer::OptimizerConfig.name']}"
)
print(
    f"      learning_rate: {study.best_params['TrainingConfig.optimizer::OptimizerConfig.learning_rate']:.6f}"
)
print(
    f"      weight_decay: {study.best_params['TrainingConfig.optimizer::OptimizerConfig.weight_decay']:.4f}"
)
print("\n   Training:")
print(f"      batch_size: {study.best_params['TrainingConfig.batch_size']}")

print("\n" + "=" * 60)
print("\n🌳 Notice the hierarchical parameter names:")
print("   'TrainingConfig.model::ModelConfig.num_layers'")
print("   'TrainingConfig.optimizer::OptimizerConfig.learning_rate'")
print("\n   This structure:")
print("   ✅ Prevents naming conflicts between nested configs")
print("   ✅ Makes it clear which config each parameter belongs to")
print("   ✅ Works automatically - no manual namespace management needed")

[I 2025-10-26 21:49:32,502] A new study created in memory with name: no-name-bb4fee75-507e-454e-9cf7-fa959df22027
[I 2025-10-26 21:49:32,508] Trial 0 finished with value: 0.7 and parameters: {'TrainingConfig.model::ModelConfig.num_layers': 6, 'TrainingConfig.model::ModelConfig.hidden_dim': 107, 'TrainingConfig.model::ModelConfig.activation': 'relu', 'TrainingConfig.optimizer::OptimizerConfig.name': 'adam', 'TrainingConfig.optimizer::OptimizerConfig.learning_rate': 0.00011810575055143691, 'TrainingConfig.optimizer::OptimizerConfig.weight_decay': 0.09096948856639085, 'TrainingConfig.batch_size': 67}. Best is trial 0 with value: 0.7.
[I 2025-10-26 21:49:32,512] Trial 1 finished with value: 0.5 and parameters: {'TrainingConfig.model::ModelConfig.num_layers': 6, 'TrainingConfig.model::ModelConfig.hidden_dim': 231, 'TrainingConfig.model::ModelConfig.activation': 'relu', 'TrainingConfig.optimizer::OptimizerConfig.name': 'sgd', 'TrainingConfig.optimizer::OptimizerConfig.learning_rate': 2.31028

🔍 Optimizing nested configuration...



[I 2025-10-26 21:49:32,721] Trial 14 finished with value: 0.75 and parameters: {'TrainingConfig.model::ModelConfig.num_layers': 3, 'TrainingConfig.model::ModelConfig.hidden_dim': 154, 'TrainingConfig.model::ModelConfig.activation': 'gelu', 'TrainingConfig.optimizer::OptimizerConfig.name': 'adam', 'TrainingConfig.optimizer::OptimizerConfig.learning_rate': 1.0306397113838472e-05, 'TrainingConfig.optimizer::OptimizerConfig.weight_decay': 0.014084734407902606, 'TrainingConfig.batch_size': 125}. Best is trial 10 with value: 0.9.
[I 2025-10-26 21:49:32,759] Trial 15 finished with value: 0.75 and parameters: {'TrainingConfig.model::ModelConfig.num_layers': 4, 'TrainingConfig.model::ModelConfig.hidden_dim': 197, 'TrainingConfig.model::ModelConfig.activation': 'gelu', 'TrainingConfig.optimizer::OptimizerConfig.name': 'adam', 'TrainingConfig.optimizer::OptimizerConfig.learning_rate': 4.026768574072816e-05, 'TrainingConfig.optimizer::OptimizerConfig.weight_decay': 0.03744721696659584, 'TrainingCo

✅ Best score: 0.9000

📊 Best nested configuration:

   Model:
      num_layers: 4
      hidden_dim: 180
      activation: gelu

   Optimizer:
      name: adam
      learning_rate: 0.000012
      weight_decay: 0.0018

   Training:
      batch_size: 100


🌳 Notice the hierarchical parameter names:
   'TrainingConfig.model::ModelConfig.num_layers'
   'TrainingConfig.optimizer::OptimizerConfig.learning_rate'

   This structure:
   ✅ Prevents naming conflicts between nested configs
   ✅ Makes it clear which config each parameter belongs to
   ✅ Works automatically - no manual namespace management needed


In [4]:
# Polymorphic fields - different config types for the same field
class AdamOptimizerConfig(sp.Config):
    """Adam-specific optimizer configuration."""

    learning_rate: float = sp.Float(ge=1e-5, le=1e-2, distribution="log")
    beta1: float = sp.Float(ge=0.8, le=0.99)
    beta2: float = sp.Float(ge=0.9, le=0.999)
    weight_decay: float = sp.Float(ge=0.0, le=0.1)


class SGDOptimizerConfig(sp.Config):
    """SGD-specific optimizer configuration."""

    learning_rate: float = sp.Float(ge=1e-4, le=1e-1, distribution="log")
    momentum: float = sp.Float(ge=0.0, le=0.99)
    nesterov: bool = sp.Categorical([True, False])


class RMSpropOptimizerConfig(sp.Config):
    """RMSprop-specific optimizer configuration."""

    learning_rate: float = sp.Float(ge=1e-5, le=1e-2, distribution="log")
    alpha: float = sp.Float(ge=0.9, le=0.999)
    momentum: float = sp.Float(ge=0.0, le=0.5)


class PolymorphicConfig(sp.Config):
    """Configuration with polymorphic optimizer field."""

    # This field can be ANY of the three optimizer types!
    optimizer: AdamOptimizerConfig | SGDOptimizerConfig | RMSpropOptimizerConfig

    batch_size: int = sp.Int(ge=16, le=128)
    num_epochs: int = sp.Int(ge=5, le=50)


def polymorphic_objective(trial):
    """Objective with polymorphic configuration."""
    config = PolymorphicConfig.from_trial(trial)

    # Simulate training - different optimizers have different optimal regions
    score = 0.5

    # Adam performs best with specific settings
    if isinstance(config.optimizer, AdamOptimizerConfig):
        score += 0.2 if 1e-4 < config.optimizer.learning_rate < 1e-3 else 0
        score += 0.1 if config.optimizer.beta1 > 0.9 else 0
        score += 0.1 if config.optimizer.weight_decay < 0.01 else 0

    # SGD with momentum is competitive
    elif isinstance(config.optimizer, SGDOptimizerConfig):
        score += 0.15 if config.optimizer.momentum > 0.8 else 0
        score += 0.1 if config.optimizer.nesterov else 0
        score += 0.15 if 1e-3 < config.optimizer.learning_rate < 1e-2 else 0

    # RMSprop works but is less optimal
    elif isinstance(config.optimizer, RMSpropOptimizerConfig):
        score += 0.1 if config.optimizer.alpha > 0.95 else 0
        score += 0.1 if config.optimizer.momentum > 0.2 else 0

    return score


# Run optimization
print("🔍 Optimizing polymorphic configuration...\n")

study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=300)
)
study.optimize(polymorphic_objective, n_trials=50, show_progress_bar=False)

print(f"✅ Best score: {study.best_value:.4f}\n")

# Determine which optimizer type was selected
optimizer_key = "PolymorphicConfig.optimizer"
optimizer_type = None
for key in study.best_params:
    if key == optimizer_key:
        optimizer_type = study.best_params[key]
        break

print("📊 Best configuration:")
print(f"   Optimizer type: {optimizer_type}")
print(f"   batch_size: {study.best_params['PolymorphicConfig.batch_size']}")
print(f"   num_epochs: {study.best_params['PolymorphicConfig.num_epochs']}")

print("\n   Optimizer parameters:")
# Display optimizer-specific parameters
for key, value in study.best_params.items():
    if "::" in key and optimizer_type in key:
        param_name = key.split(".")[-1]
        if isinstance(value, float):
            print(f"      {param_name}: {value:.6f}")
        else:
            print(f"      {param_name}: {value}")

print("\n" + "=" * 60)
print("\n🎭 Polymorphic field analysis:")

# Count how many times each optimizer type was tried
optimizer_counts = {
    "AdamOptimizerConfig": 0,
    "SGDOptimizerConfig": 0,
    "RMSpropOptimizerConfig": 0,
}
for trial in study.trials:
    opt_type = trial.params.get(optimizer_key)
    if opt_type:
        optimizer_counts[opt_type] += 1

print("   Trials per optimizer type:")
for opt_type, count in optimizer_counts.items():
    print(f"      {opt_type}: {count} trials")

print("\n   ✅ Optuna explored different config types automatically!")
print("   ✅ Each type has its own unique parameter space")
print(f"   ✅ Best optimizer: {optimizer_type}")

[I 2025-10-26 21:50:50,435] A new study created in memory with name: no-name-6f92df10-3b2d-490d-9e47-b04497b6ef17
[I 2025-10-26 21:50:50,440] Trial 0 finished with value: 0.5 and parameters: {'PolymorphicConfig.optimizer': 'AdamOptimizerConfig', 'PolymorphicConfig.optimizer::AdamOptimizerConfig.learning_rate': 7.4529311852234e-05, 'PolymorphicConfig.optimizer::AdamOptimizerConfig.beta1': 0.8458048062255319, 'PolymorphicConfig.optimizer::AdamOptimizerConfig.beta2': 0.9774225968630594, 'PolymorphicConfig.optimizer::AdamOptimizerConfig.weight_decay': 0.08110943659648512, 'PolymorphicConfig.batch_size': 61, 'PolymorphicConfig.num_epochs': 6}. Best is trial 0 with value: 0.5.
[I 2025-10-26 21:50:50,444] Trial 1 finished with value: 0.7 and parameters: {'PolymorphicConfig.optimizer': 'RMSpropOptimizerConfig', 'PolymorphicConfig.optimizer::RMSpropOptimizerConfig.learning_rate': 0.003765072225488711, 'PolymorphicConfig.optimizer::RMSpropOptimizerConfig.alpha': 0.9580934719593732, 'PolymorphicC

🔍 Optimizing polymorphic configuration...



[I 2025-10-26 21:50:50,641] Trial 20 finished with value: 0.5 and parameters: {'PolymorphicConfig.optimizer': 'SGDOptimizerConfig', 'PolymorphicConfig.optimizer::SGDOptimizerConfig.learning_rate': 0.0008752831799984179, 'PolymorphicConfig.optimizer::SGDOptimizerConfig.momentum': 0.7381823796480853, 'PolymorphicConfig.optimizer::SGDOptimizerConfig.nesterov': 'False', 'PolymorphicConfig.batch_size': 112, 'PolymorphicConfig.num_epochs': 9}. Best is trial 7 with value: 0.8.
[I 2025-10-26 21:50:50,661] Trial 21 finished with value: 0.8 and parameters: {'PolymorphicConfig.optimizer': 'SGDOptimizerConfig', 'PolymorphicConfig.optimizer::SGDOptimizerConfig.learning_rate': 0.0028042567144470074, 'PolymorphicConfig.optimizer::SGDOptimizerConfig.momentum': 0.9875051326012375, 'PolymorphicConfig.optimizer::SGDOptimizerConfig.nesterov': 'False', 'PolymorphicConfig.batch_size': 103, 'PolymorphicConfig.num_epochs': 13}. Best is trial 7 with value: 0.8.
[I 2025-10-26 21:50:50,685] Trial 22 finished wit

✅ Best score: 0.9000

📊 Best configuration:
   Optimizer type: SGDOptimizerConfig
   batch_size: 16
   num_epochs: 17

   Optimizer parameters:
      learning_rate: 0.002992
      momentum: 0.910263
      nesterov: True


🎭 Polymorphic field analysis:
   Trials per optimizer type:
      AdamOptimizerConfig: 8 trials
      SGDOptimizerConfig: 34 trials
      RMSpropOptimizerConfig: 8 trials

   ✅ Optuna explored different config types automatically!
   ✅ Each type has its own unique parameter space
   ✅ Best optimizer: SGDOptimizerConfig


In [ ]:
# Complete example: All features together
class CNNConfig(sp.Config):
    """CNN encoder configuration."""

    num_conv_layers: int = sp.Int(ge=2, le=4)
    num_filters: int = sp.Int(ge=32, le=128)
    kernel_size: int = sp.Categorical([3, 5])


class TransformerConfig(sp.Config):
    """Transformer encoder configuration."""

    num_layers: int = sp.Int(ge=2, le=6)
    num_heads: int = sp.Int(ge=2, le=8)
    hidden_dim: int = sp.Int(ge=128, le=512)


class CompleteConfig(sp.Config):
    """Complete configuration combining all SpaX features."""

    # Polymorphic: Different encoder architectures
    encoder: CNNConfig | TransformerConfig

    # Polymorphic: Different optimizers
    optimizer: AdamOptimizerConfig | SGDOptimizerConfig

    # Regular parameters
    batch_size: int = sp.Int(ge=16, le=128)
    use_augmentation: bool = sp.Categorical([True, False])

    # Conditional: Augmentation strength only when augmentation is enabled
    augmentation_strength: float = sp.Conditional(
        sp.FieldCondition("use_augmentation", sp.EqualsTo(True)),
        true=sp.Float(ge=0.1, le=0.9),
        false=0.0,
    )

    # Conditional: Gradient clipping based on encoder type
    gradient_clip: float = sp.Conditional(
        sp.FieldCondition("encoder", sp.IsInstance(TransformerConfig)),
        true=sp.Float(ge=0.5, le=5.0),  # Transformers need clipping
        false=1.0,  # CNNs typically don't
    )


def complete_objective(trial):
    """
    Complete objective function showing best practices.

    In a real scenario, this would:
    1. Create config from trial
    2. Initialize model with config
    3. Train and validate
    4. Return validation metric
    """
    config = CompleteConfig.from_trial(trial)

    # Simulate complex scoring
    score = 0.4

    # Encoder type matters
    if isinstance(config.encoder, TransformerConfig):
        score += 0.15 if config.encoder.num_heads == 4 else 0
        score += 0.1 if 256 <= config.encoder.hidden_dim <= 384 else 0
        score += 0.05 if config.gradient_clip > 1.0 else 0
    else:  # CNN
        score += 0.1 if config.encoder.num_conv_layers == 3 else 0
        score += 0.1 if config.encoder.num_filters >= 64 else 0

    # Optimizer matters
    if isinstance(config.optimizer, AdamOptimizerConfig):
        score += 0.15 if 1e-4 < config.optimizer.learning_rate < 5e-4 else 0
    else:  # SGD
        score += 0.1 if config.optimizer.momentum > 0.85 else 0

    # Augmentation helps
    score += (
        0.1
        if config.use_augmentation and 0.3 < config.augmentation_strength < 0.7
        else 0
    )

    return score


# Best practice: Use logging to track progress
print("🏭 Running complete optimization with all SpaX features...\n")
print("Features being optimized:")
print("  ✅ Nested configs (encoder has its own parameters)")
print("  ✅ Polymorphic fields (different encoder & optimizer types)")
print("  ✅ Conditional parameters (depend on other choices)")
print("  ✅ Type constraints (Transformer → gradient clipping)")
print()

# Create study with logging
optuna.logging.set_verbosity(optuna.logging.WARNING)  # Reduce noise

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=400),
    study_name="spax_complete_optimization",
)

study.optimize(complete_objective, n_trials=60, show_progress_bar=False)

print("✅ Optimization complete!")
print(f"   Best score: {study.best_value:.4f}")
print(f"   Total trials: {len(study.trials)}")
print()

# Reconstruct the best config
best_config = CompleteConfig.from_trial(study.best_trial)
encoder_type = type(best_config.encoder).__name__
optimizer_type = type(best_config.optimizer).__name__

print("🏆 Best configuration found:")
print(f"   Encoder: {encoder_type}")
print(f"   Optimizer: {optimizer_type}")
print(f"   Batch size: {best_config.batch_size}")
print(f"   Augmentation: {best_config.use_augmentation}")
if best_config.use_augmentation:
    print(f"   Aug strength: {best_config.augmentation_strength:.3f}")
print(f"   Gradient clip: {best_config.gradient_clip:.3f}")

print("\n" + "=" * 60)
print("\n💡 Best Practice Workflow:")
print("   1. Define config with all SpaX features you need")
print("   2. Create objective: config = MyConfig.from_trial(trial)")
print("   3. Run optimization: study.optimize(objective, n_trials=N)")
print("   4. Retrieve best: best = MyConfig.from_trial(study.best_trial)")
print("   5. Use Optuna's tools for visualization and analysis")

🏭 Running complete optimization with all SpaX features...

Features being optimized:
  ✅ Nested configs (encoder has its own parameters)
  ✅ Polymorphic fields (different encoder & optimizer types)
  ✅ Conditional parameters (depend on other choices)
  ✅ Type constraints (Transformer → gradient clipping)

✅ Optimization complete!
   Best score: 0.8500
   Total trials: 60

🏆 Best configuration found:
   Encoder: TransformerConfig
   Optimizer: AdamOptimizerConfig
   Batch size: 55
   Augmentation: True
   Aug strength: 0.568
   Gradient clip: 3.531


💡 Best Practice Workflow:
   1. Define config with all SpaX features you need
   2. Create objective: config = MyConfig.from_trial(trial)
   3. Run optimization: study.optimize(objective, n_trials=N)
   4. Retrieve best: best = MyConfig.from_trial(study.best_trial)
   5. Use Optuna's tools for visualization and analysis


## 📝 Summary: SpaX + Optuna Integration

You've learned how SpaX integrates seamlessly with Optuna for hyperparameter optimization.

### ✅ Key Integration Pattern

**One line is all you need:**
```python
def objective(trial):
    config = MyConfig.from_trial(trial)
    score = train_and_evaluate(config)
    return score
```

That's it! SpaX handles:
- ✅ Suggesting values from all parameter spaces
- ✅ Respecting conditional dependencies
- ✅ Managing nested config hierarchies
- ✅ Handling polymorphic fields (Union types)
- ✅ Hierarchical parameter naming (prevents conflicts)

### ✅ All SpaX Features Work Automatically

1. **Conditional Parameters**
   - Optuna never suggests invalid combinations
   - Dependencies are handled automatically
   - No manual constraint management needed

2. **Nested Configs**
   - All nested parameters are optimized
   - Hierarchical naming: `ParentConfig.child::ChildConfig.param`
   - No namespace conflicts

3. **Polymorphic Fields (Union Types)**
   - Optuna explores different config types
   - Each type has its own parameter space
   - Type selection is part of optimization

### ✅ Best Practices

**1. Define your config with SpaX:**
```python
class MyConfig(sp.Config):
    param1: int = sp.Int(ge=1, le=10)
    param2: float = sp.Float(ge=0.0, le=1.0)
    # ... with conditionals, nesting, polymorphism as needed
```

**2. Create objective function:**
```python
def objective(trial):
    config = MyConfig.from_trial(trial)
    return train_model(config)  # Your training code
```

**3. Run optimization:**
```python
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)
```

**4. Retrieve best config:**
```python
best_config = MyConfig.from_trial(study.best_trial)
```

**5. Use Optuna's analysis tools:**
```python
# Optuna's built-in visualizations and analysis
optuna.visualization.plot_optimization_history(study)
optuna.visualization.plot_param_importances(study)
```

### 🎯 Key Takeaways

- **Zero boilerplate**: `from_trial()` handles everything
- **Type-safe**: Invalid configs never get suggested
- **Scalable**: Works with arbitrarily complex config structures
- **Standard Optuna**: Use all Optuna features (pruning, samplers, parallel, etc.)
- **Hierarchical naming**: Automatic conflict prevention

### 🚀 What's Next?

- **Notebook 05**: Iterative refinement with the override system
- **Real HPO**: Apply this to your actual ML training pipeline
- **Advanced Optuna**: Combine with pruning, multi-objective, distributed optimization

**You now have powerful, type-safe hyperparameter optimization! 🎉**